# Lane D · Shopper history

Use all available history to find changes Gabriel and Eudald can test in the recommender.
The practice section counts requests using the competition's cold-request rule.

From the repository root, launch Jupyter with:\n\n```powershell\nuv run --with jupyterlab jupyter lab notebooks/miles-history-eda.ipynb\n```\n\nRun all cells using the Python 3 kernel. The team credentials load from the local `.env`.\nThe queries read RawTree and return aggregate results. All timestamps shown are UTC.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

from dades_huzz.data import RawTree

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
rawtree = RawTree.from_env(root / ".env")


def query(sql):
    return rawtree.query(sql, timeout=60)


def show_table(rows):
    if not rows:
        display(Markdown("No matching records."))
        return

    columns = list(rows[0])

    def format_row(values):
        cells = [str(value).replace("|", "&#124;") if value is not None else "—" for value in values]
        return "| " + " | ".join(cells) + " |"

    lines = [format_row(columns), format_row(["---"] * len(columns))]
    lines.extend(format_row(row[column] for column in columns) for row in rows)
    display(Markdown("\n".join(lines)))

## 1. How did each carousel position perform?

The rate at a position uses every time a product was shown there as its denominator. The old
carousel may have put stronger products in certain positions. The second table compares positions
1 and 5 using the same products, weighted equally. It checks whether product mix could explain
some of the gap. Different shoppers and situations can still affect the comparison.

In [ ]:
slot_performance = query("""
WITH impressions AS (
    SELECT
        shown::Array(String) AS products,
        coalesce(clicked::Nullable(String), '') AS click_product,
        coalesce(purchased::Nullable(String), '') AS purchase_product
    FROM shop.history_sessions
)
SELECT
    slot,
    count() AS times_shown,
    countIf(products[slot] = click_product AND click_product != '') AS clicks,
    countIf(products[slot] = purchase_product AND purchase_product != '') AS purchases,
    round(100.0 * clicks / times_shown, 3) AS clicks_per_100_shows,
    round(100.0 * purchases / times_shown, 3) AS purchases_per_100_shows
FROM impressions
ARRAY JOIN arrayEnumerate(products) AS slot
GROUP BY slot
ORDER BY slot
""")
show_table(slot_performance)

same_products = query("""
WITH impressions AS (
    SELECT
        shown::Array(String) AS products,
        coalesce(purchased::Nullable(String), '') AS purchase_product
    FROM shop.history_sessions
    WHERE length(shown::Array(String)) >= 5
), by_product AS (
    SELECT
        products[slot] AS product,
        countIf(slot = 1) AS first_shows,
        countIf(slot = 5) AS fifth_shows,
        countIf(slot = 1 AND product = purchase_product) AS first_purchases,
        countIf(slot = 5 AND product = purchase_product) AS fifth_purchases
    FROM impressions
    ARRAY JOIN [1, 5] AS slot
    GROUP BY product
    HAVING first_shows >= 100 AND fifth_shows >= 100
)
SELECT
    count() AS products_compared,
    sum(first_shows) AS slot_1_shows,
    sum(fifth_shows) AS slot_5_shows,
    round(100.0 * sum(first_purchases / first_shows) / nullIf(count(), 0), 3) AS slot_1_rate,
    round(100.0 * sum(fifth_purchases / fifth_shows) / nullIf(count(), 0), 3) AS slot_5_rate
FROM by_product
""")
show_table(same_products)

## 2. How often did the historical carousel have no recent views?

This is the share of historical requests with no recorded products viewed earlier in the visit.
It measures when a recent-view signal is unavailable. The practice section measures the official
cold-request share.

In [ ]:
history = query("""
SELECT
    toString(min(ts::DateTime64(3))) AS first_request_utc,
    toString(max(ts::DateTime64(3))) AS last_request_utc,
    count() AS requests,
    countIf(empty(recent_views::Array(Dynamic))) AS requests_without_recent_views,
    countIf(`context.page` = 'home' AND empty(recent_views::Array(Dynamic))) AS home_without_recent_views,
    countIf(`context.page` = 'home' AND notEmpty(recent_views::Array(Dynamic))) AS home_with_recent_views,
    round(100.0 * requests_without_recent_views / nullIf(requests, 0), 3) AS pct_without_recent_views,
    countIf(coalesce(purchased::Nullable(String), '') != '') AS purchases
FROM shop.history_sessions
""")
show_table(history)

## 3. How did purchases vary by page, device, and region?

Each breakdown uses the number of requests in that group. These are separate comparisons.
For example, the device rates also reflect the mix of pages and shoppers using each device.

In [ ]:
conversion = query("""
WITH impressions AS (
    SELECT
        coalesce(`context.page`::Nullable(String), '') AS page,
        coalesce(`context.device`::Nullable(String), '') AS device,
        coalesce(`context.region`::Nullable(String), '') AS region,
        coalesce(purchased::Nullable(String), '') != '' AS bought
    FROM shop.history_sessions
)
SELECT
    segment.1 AS dimension,
    if(empty(segment.2), '(missing)', segment.2) AS segment_value,
    count() AS requests,
    countIf(bought) AS purchases,
    round(100.0 * purchases / requests, 3) AS purchases_per_100_requests
FROM impressions
ARRAY JOIN [('Page', page), ('Device', device), ('Region', region)] AS segment
GROUP BY dimension, segment_value
ORDER BY dimension, requests DESC, segment_value
""")
show_table(conversion)

## 4. How much demand went to the top 10 products?

Rank products by purchase count separately in regular store activity and the historical carousel.
Show the share of purchases going to each source's own top 10, overall and within each category.
Categories use the starting catalogue. Unmatched products appear under `(unknown)`.
The two sources had different exposure, so the comparison can suggest where to test a wider range.

In [ ]:
concentration = query("""
WITH purchases_by_product AS (
    SELECT product_id::String AS product, 'Organic' AS source, count() AS purchases
    FROM shop.history_events
    WHERE event_type = 'purchase'
    GROUP BY product
    UNION ALL
    SELECT purchased::String AS product, 'Carousel' AS source, count() AS purchases
    FROM shop.history_sessions
    WHERE coalesce(purchased::Nullable(String), '') != ''
    GROUP BY product
), category_totals AS (
    SELECT source, product, purchases,
           coalesce(nullIf(c.category::String, ''), '(unknown)') AS product_category
    FROM purchases_by_product AS p
    LEFT ANY JOIN shop.catalog AS c ON p.product = c.product_id::String
), ranked AS (
    SELECT source, scope,
           purchases,
           row_number() OVER (PARTITION BY source, scope ORDER BY purchases DESC, product) AS rank
    FROM category_totals
    ARRAY JOIN ['All products', product_category] AS scope
)
SELECT
    scope, source, sum(purchases) AS total_purchases, count() AS products_purchased,
    sumIf(purchases, rank <= 10) AS top_10_purchases,
    round(100.0 * top_10_purchases / total_purchases, 3) AS top_10_share_pct
FROM ranked
GROUP BY scope, source
ORDER BY scope != 'All products', scope, source DESC
""")
show_table(concentration)

## 5. How often did shoppers buy the same product on a later visit?

Use regular store purchases because those records identify the visit with a session ID.
A purchase counts as a repeat when the shopper bought the same product in another, earlier
session in the available history. Events in the same session count as purchases but do not
establish an earlier visit. Views and add-to-cart events do not count as purchases.
Include the number of purchases with usable shopper, product, and session IDs.

In [ ]:
repeat_coverage = query("""
SELECT
    toString(min(ts::DateTime64(3))) AS first_purchase_utc,
    toString(max(ts::DateTime64(3))) AS last_purchase_utc,
    count() AS organic_purchases,
    countIf(user_id::String = '' OR session_id::String = '' OR product_id::String = '') AS missing_ids
FROM shop.history_events
WHERE event_type = 'purchase'
""")
show_table(repeat_coverage)

repeat_purchases = query("""
WITH session_purchases AS (
    SELECT
        user_id::String AS shopper, product_id::String AS product,
        session_id::String AS session, min(ts::DateTime64(9)) AS first_buy,
        count() AS purchases
    FROM shop.history_events
    WHERE event_type = 'purchase'
        AND user_id::String != '' AND product_id::String != '' AND session_id::String != ''
    GROUP BY shopper, product, session
), marked AS (
    SELECT product, purchases, first_buy >
        min(first_buy) OVER (PARTITION BY shopper, product) AS is_repeat
    FROM session_purchases
), product_totals AS (
    SELECT product, sum(purchases) AS product_purchases, sumIf(purchases, is_repeat) AS repeats
    FROM marked
    GROUP BY product
)
SELECT
    scope, sum(product_purchases) AS purchases_with_ids, sum(repeats) AS repeat_purchases,
    round(100.0 * repeat_purchases / purchases_with_ids, 3) AS repeat_purchase_pct
FROM product_totals AS p
LEFT ANY JOIN shop.catalog AS c ON p.product = c.product_id::String
ARRAY JOIN ['All products', coalesce(nullIf(c.category::String, ''), '(unknown)')] AS scope
GROUP BY scope
ORDER BY scope != 'All products', purchases_with_ids DESC, scope
""")
show_table(repeat_purchases)

## Practice only · How many requests were officially cold?

Use all practice requests available at query time, from Saturday 09:00 UTC up to Sunday
09:00 UTC (10:00 CET on both days). Show the first and last requests actually included.
A request is cold when the shopper is new or recently viewed a product introduced during
the event. Count the two reasons separately and count their overlap once.

In [ ]:
practice = query("""
WITH
    toDateTime64('2026-11-14 09:00:00', 9, 'UTC') AS practice_start,
    least(now64(9, 'UTC'), toDateTime64('2026-11-15 09:00:00', 9, 'UTC')) AS practice_end,
    launches AS (
        SELECT DISTINCT product_id::String AS product
        FROM shop.catalog_updates
        WHERE change = 'new_product'
            AND ts::DateTime64(9) >= practice_start AND ts::DateTime64(9) < practice_end
    ),
    requests AS (
        SELECT
            ts::DateTime64(9) AS request_time,
            coalesce(is_new_user::Nullable(Bool), false) AS new_shopper,
            arrayExists(
                view -> JSONExtractString(toJSONString(view), 'product_id') IN (SELECT product FROM launches),
                recent_views::Array(Dynamic)
            ) AS viewed_new_product
        FROM shop.requests
        WHERE request_time >= practice_start AND request_time < practice_end
    )
SELECT
    toString(minOrNull(request_time)) AS first_request_utc,
    toString(maxOrNull(request_time)) AS last_request_utc,
    count() AS requests,
    countIf(new_shopper AND NOT viewed_new_product) AS new_shopper_only,
    countIf(viewed_new_product AND NOT new_shopper) AS viewed_new_product_only,
    countIf(new_shopper AND viewed_new_product) AS both,
    countIf(new_shopper OR viewed_new_product) AS cold_requests,
    round(100.0 * cold_requests / nullIf(requests, 0), 3) AS cold_share_pct
FROM requests
""")
show_table(practice)

## Five findings to test

Snapshot from 14 November 2026. The historical tables cover all 60 available days. Rerun the
practice cell to include any later practice requests.

1. **Put the strongest candidates first.** Slot 1 had 1.074 clicks and 0.449 purchases per 100
   shows, compared with 0.281 clicks and 0.100 purchases in slot 5 (2 million shows per slot).
   The purchase gap remained when the same 977 products were weighted equally across the two
   slots (0.081 vs 0.034 per 100 shows). Try ranking the same candidates by estimated purchase
   likelihood and compare against the current order in practice.
2. **Test how the model handles requests with little history.** Historically, 400,549 of 2 million
   requests (20.03%) had no recent views, and all were home-page requests. In practice, from
   09:00:00 to 13:05:20 UTC on 14 November, 14,243 of 44,161 requests (32.25%) were officially cold.
   There were 12,843 new-shopper-only requests, 999 new-product-view-only requests, and 401 that
   met both conditions. Test a popularity-based default for empty visits and catalogue similarity
   for newly viewed products.
3. **Measure home-page lift separately.** The old carousel got 0.145 purchases per 100 home-page
   requests (580 of 400,549), versus 1.721 on product pages (19,263 of 1,119,265) and 1.710 on category
   pages (8,210 of 480,186). Compare a home-page default that uses recent store demand with the
   global best-seller list. Breakdowns by device and region are in the table above.
4. **Try a wider candidate list.** The top 10 products accounted for 7.20% of 548,160 regular store
   purchases and 51.70% of 28,053 purchases from the old carousel. In Pets, the corresponding shares
   were 17.56% and 71.31%. Test adding products drawn from regular store activity to the candidates
   for categories such as Pets, where the carousel concentrated purchases much more heavily.
5. **Test how strongly to lower the rank of previously bought products.** 79,638 of 548,160 organic
   purchases (14.53%) repeated the same shopper and product in a later session within the available
   history. The share was 19.65% in Beauty & Health (7,251 of 36,899) and 6.25% in Home & Garden
   (1,184 of 18,934). Compare a smaller rank penalty in higher-repeat categories with a single
   penalty across all categories.

These are observations under the historical shopping and carousel patterns. The proposed
changes need a separate comparison to establish whether they improve purchases.